# Lab: prevendo a sobrevivência no Titanic

Notebook de aula da página *Regressão Logística*: https://insper.github.io/ML/pt/2026.2/classes/logistic-regression/

A lista de passageiros do Titanic da aula de [pré-processamento](../preprocessing/index.md), agora com uma pergunta: *quem sobreviveu?* O lab aplica tudo o que foi visto a dados reais e bagunçados: um pipeline com imputação, codificação e escalonamento; probabilidades conferidas à mão; coeficientes lidos como razões de chances; e uma estimativa honesta da acurácia em passageiros que o modelo nunca viu. O modelo é a regressão logística pura desta aula (`C=np.inf`), então não há nada para ajustar: dividimos os dados uma vez em **treino** (ajustar e estudar o modelo) e **teste** (reportar, uma vez, no fim).

Rode as células em ordem. Em cada passo, responda o **Preveja** com a dupla *antes* de rodar. Nas **Simulações**, escreva o experimento na célula vazia; as soluções e a leitura de cada resultado estão na página da aula.

## Passo 1 — carregar e olhar

**Preveja.** A taxa de sobrevivência geral fica acima ou abaixo de 50%? Quem sobreviveu mais: as mulheres da 3ª classe ou os homens da 1ª?

In [1]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/hsandmann/biblio/refs/heads/main/ml/aula04/titanic/train.csv"
df = pd.read_csv(url)
print(df.shape)
print(f"taxa de sobrevivência: {df['Survived'].mean():.3f}")
print(df.pivot_table(index='Sex', columns='Pclass', values='Survived', aggfunc='mean').round(3))
print("faltantes:", df[['Age', 'Embarked', 'Cabin']].isna().sum().to_dict())

(891, 12)
taxa de sobrevivência: 0.384
Pclass      1      2      3
Sex                        
female  0.968  0.921  0.500
male    0.369  0.157  0.135
faltantes: {'Age': 177, 'Embarked': 2, 'Cabin': 687}


**Resposta.** Abaixo: a taxa geral é 38,4%. As mulheres da 3ª classe (50,0%) sobreviveram bem mais que os homens da 1ª (36,9%): o sexo pesou mais que a classe.

## Passo 2 — treino e teste

**Preveja.** Por que separar o conjunto de teste agora, antes de montar qualquer modelo? Por que não deixar isso para o fim?

In [2]:
from sklearn.model_selection import train_test_split

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X, y = df[features], df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(len(X_train), len(X_test))
print(f"taxa de sobrevivência: treino {y_train.mean():.3f} · teste {y_test.mean():.3f}")

712 179
taxa de sobrevivência: treino 0.383 · teste 0.385


**Resposta.** Para que nenhuma decisão (pré-processamento, atributos, olhar os dados) seja influenciada pelo teste; deixado para o fim, ele já teria vazado para as escolhas e a estimativa deixaria de ser honesta. A divisão estratificada mantém a taxa quase igual nas duas partes: 0,383 no treino (712 linhas) e 0,385 no teste (179).

## Passo 3 — uma baseline

**Preveja.** Um "modelo" que responde sempre "não sobreviveu", sem olhar o passageiro, acerta mais ou menos da metade das vezes? Por quê?

In [3]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print(f"sempre 'não sobreviveu': acurácia de treino {dummy.score(X_train, y_train):.3f}")

sempre 'não sobreviveu': acurácia de treino 0.617


**Resposta.** Mais da metade: acerta 61,7% no treino, porque 61,7% dos passageiros morreram e responder sempre a classe majoritária acerta exatamente essa fração. É o piso que qualquer modelo precisa superar.

## Passo 4 — o pipeline

**Preveja.** `Pclass` vale 1, 2 ou 3. Por que tratá-la como categoria (uma coluna one-hot para cada classe) e não como um número?

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = ['Age', 'SibSp', 'Parch', 'Fare']
categorical = ['Pclass', 'Sex', 'Embarked']

preprocess = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(drop='first')), categorical)])
model = make_pipeline(preprocess, LogisticRegression(C=np.inf, max_iter=1000))   # C=np.inf: sem penalidade
model.fit(X_train, y_train)

enc = model[0].named_transformers_['cat'][-1]
print("categorias de referência (descartadas):", {c: cats[0] for c, cats in zip(categorical, enc.categories_)})
print(f"{model[-1].coef_.size} coeficientes · acurácia de treino {model.score(X_train, y_train):.3f}")

categorias de referência (descartadas): {'Pclass': 1, 'Sex': 'female', 'Embarked': 'C'}
9 coeficientes · acurácia de treino 0.809


**Resposta.** Como número, `Pclass` teria um único peso e o modelo seria forçado a supor que ir da 1ª para a 2ª classe muda o escore tanto quanto ir da 2ª para a 3ª; com one-hot, cada classe ganha seu próprio efeito em relação à referência (1ª classe). O pipeline ficou com 9 coeficientes e acurácia de treino de 0,809.

## Passo 5 — probabilidades para dois passageiros

**Preveja.** Olhando a tabela do passo 1: um homem da 3ª classe e uma mulher da 1ª classe. Quem recebe probabilidade alta e quem recebe baixa?

In [5]:
passengers = pd.DataFrame({'Pclass': [3, 1], 'Sex': ['male', 'female'], 'Age': [25, 30],
                           'SibSp': [0, 1], 'Parch': [0, 0], 'Fare': [8.05, 80.0],
                           'Embarked': ['S', 'C']})
print(model.predict_proba(passengers).round(3))
print(model.predict(passengers))

[[0.907 0.093]
 [0.058 0.942]]
[0 1]


**Resposta.** Como sugere a tabela do passo 1, o homem da 3ª classe recebe probabilidade baixa (0,093) e a mulher da 1ª classe, alta (0,942); o `predict` devolve 0 e 1.

**Simulação: o efeito da idade.** Tome um passageiro da 2ª classe que viaja com os pais (`Parch = 2`) e um irmão (`SibSp = 1`), passagem de 30 libras, embarcado em Southampton. Mude só a idade (2, 8, 15, 25, 40 e 60 anos), primeiro para um menino e depois para uma menina. Observe a probabilidade prevista. Depois confira nas linhas de treino quantas crianças de até 12 anos da 2ª classe sobreviveram.

In [6]:
ages = [2, 8, 15, 25, 40, 60]
print("idade  ", ages)
for sex in ['male', 'female']:
    people = pd.DataFrame({'Pclass': 2, 'Sex': sex, 'Age': ages, 'SibSp': 1, 'Parch': 2,
                           'Fare': 30.0, 'Embarked': 'S'})
    print(f"{sex:7s}", model.predict_proba(people)[:, 1].round(3))

train = df.loc[X_train.index]                        # só as linhas de treino
kids = train[(train['Age'] <= 12) & (train['Pclass'] == 2)]
print(kids.groupby('Sex')['Survived'].agg(['mean', 'size']))

idade   [2, 8, 15, 25, 40, 60]
male    [0.38  0.326 0.268 0.198 0.12  0.059]
female  [0.9   0.876 0.843 0.784 0.667 0.477]
        mean  size
Sex               
female   1.0     7
male     1.0     8


**Leitura.** A probabilidade cai com a idade para os dois sexos: de 0,380 a 0,059 para o menino e de 0,900 a 0,477 para a menina; o menino de 8 anos fica em 0,326. Mas no treino todas as 15 crianças de até 12 anos da 2ª classe sobreviveram (7 meninas e 8 meninos, média 1,0). O modelo usa o mesmo efeito de sexo em todas as idades e aplica ao menino a penalidade de homem adulto: sem um termo de interação, ele não captura que crianças eram salvas independentemente do sexo.

## Passo 6 — a probabilidade do homem, à mão

**Preveja.** Olhando os coeficientes que vão aparecer, qual atributo você espera que mais derrube o escore do homem?

In [7]:
names = model[0].get_feature_names_out()
w, b = model[-1].coef_[0], model[-1].intercept_[0]
x_man = model[0].transform(passengers)[0]            # o homem da 3ª classe, depois do pré-processamento
print(pd.DataFrame({'x': x_man, 'w': w, 'w*x': w * x_man}, index=names).round(3))
z = b + (w * x_man).sum()
print(f"z = {b:.3f} + {(w * x_man).sum():.3f} = {z:.3f}   sigmoid(z) = {1 / (1 + np.exp(-z)):.3f}")

                     x      w    w*x
num__Age        -0.350 -0.513  0.180
num__SibSp      -0.465 -0.267  0.124
num__Parch      -0.466 -0.072  0.034
num__Fare       -0.495  0.108 -0.053
cat__Pclass_2    0.000 -0.987 -0.000
cat__Pclass_3    1.000 -2.239 -2.239
cat__Sex_male    1.000 -2.685 -2.685
cat__Embarked_Q  0.000  0.320  0.000
cat__Embarked_S  1.000 -0.417 -0.417
z = 2.782 + -5.057 = -2.275   sigmoid(z) = 0.093


**Resposta.** O sexo: `Sex_male` contribui com −2,685 para o escore, seguido de perto pela 3ª classe (−2,239). Somadas ao intercepto 2,782, as contribuições dão $z = -2{,}275$ e $\sigma(z) = 0{,}093$, a mesma probabilidade do `predict_proba`.

**Simulação: um atributo por vez.** Parta do homem da 3ª classe e transforme-o, um passo de cada vez, na passageira de referência: primeiro mude o sexo para `female`, depois a classe para 1, depois o porto para `C`. A cada passo, imprima o escore \(z\) (`model.decision_function`) e a probabilidade. Quanto \(z\) muda em cada passo? E a probabilidade?

In [8]:
step = passengers.iloc[[0]].copy()                   # começa no homem da 3ª classe
z_old, p_old = model.decision_function(step)[0], model.predict_proba(step)[0, 1]
print(f"{'início':22s} z = {z_old:+.3f}            p = {p_old:.3f}")
for col, val in [('Sex', 'female'), ('Pclass', 1), ('Embarked', 'C')]:
    step[col] = val
    z, p = model.decision_function(step)[0], model.predict_proba(step)[0, 1]
    print(f"{col + ' -> ' + str(val):22s} z = {z:+.3f} ({z - z_old:+.3f})  p = {p:.3f} ({p - p_old:+.3f})")
    z_old, p_old = z, p

início                 z = -2.275            p = 0.093
Sex -> female          z = +0.410 (+2.685)  p = 0.601 (+0.508)
Pclass -> 1            z = +2.649 (+2.239)  p = 0.934 (+0.333)
Embarked -> C          z = +3.066 (+0.417)  p = 0.955 (+0.022)


**Leitura.** Cada passo soma ao escore exatamente o seu coeficiente: +2,685 (sexo), +2,239 (classe) e +0,417 (porto), levando $z$ de −2,275 a +3,066. A probabilidade não acompanha de forma linear: sobe 0,508 com o sexo, 0,333 com a classe e só 0,022 com o porto, porque perto de 1 a sigmoide achata. O escore é aditivo; a probabilidade, não.

## Passo 7 — coeficientes como razões de chances

**Preveja.** O coeficiente de `Age` é negativo. Então, com o resto fixo, um passageiro mais velho tem chance maior ou menor de sobreviver? A razão de chances $e^{\text{coef}}$ dele fica acima ou abaixo de 1?

In [9]:
coef = pd.Series(w, index=names)
print(pd.DataFrame({'coef': coef, 'e^coef': np.exp(coef)}).round(3))
sd = model[0].named_transformers_['num'][-1].scale_
print("um desvio-padrão:", ", ".join(f"{c} {s:.1f}" for c, s in zip(numeric, sd)))

                  coef  e^coef
num__Age        -0.513   0.599
num__SibSp      -0.267   0.765
num__Parch      -0.072   0.931
num__Fare        0.108   1.114
cat__Pclass_2   -0.987   0.373
cat__Pclass_3   -2.239   0.107
cat__Sex_male   -2.685   0.068
cat__Embarked_Q  0.320   1.377
cat__Embarked_S -0.417   0.659
um desvio-padrão: Age 13.0, SibSp 1.1, Parch 0.8, Fare 48.0


In [10]:
print(f"mulher vs homem:  chance x {np.exp(-coef['cat__Sex_male']):.2f}")
print(f"1ª vs 3ª classe:  chance x {np.exp(-coef['cat__Pclass_3']):.2f}")
print(f"2ª vs 3ª classe:  chance x {np.exp(coef['cat__Pclass_2'] - coef['cat__Pclass_3']):.2f}")

mulher vs homem:  chance x 14.66
1ª vs 3ª classe:  chance x 9.39
2ª vs 3ª classe:  chance x 3.50


**Resposta.** Menor: com o resto fixo, cada desvio-padrão a mais de idade (13 anos) multiplica a chance de sobreviver por $e^{-0{,}513} = 0{,}599$, abaixo de 1, ou seja, reduz a chance em cerca de 40%. Na outra ponta, ser mulher multiplica a chance por 14,66 e estar na 1ª classe (contra a 3ª), por 9,39.

**Simulação: a razão de chances é a mesma para todo mundo.** Tome três passageiros diferentes (o homem da 3ª classe, um homem de 30 anos da 1ª classe e o menino da simulação do passo 5). Para cada um, calcule com `predict_proba` a probabilidade como mulher e como homem, mantendo todo o resto. Compare a razão entre as probabilidades e a razão entre as chances.

In [11]:
odds = lambda p: p / (1 - p)
people = pd.DataFrame({'Pclass': [3, 1, 2], 'Sex': 'male', 'Age': [25, 30, 8], 'SibSp': [0, 1, 1],
                       'Parch': [0, 0, 2], 'Fare': [8.05, 80.0, 30.0], 'Embarked': ['S', 'C', 'S']})
for i in range(len(people)):
    one = people.iloc[[i]]
    p_f = model.predict_proba(one.assign(Sex='female'))[0, 1]
    p_m = model.predict_proba(one.assign(Sex='male'))[0, 1]
    print(f"p mulher {p_f:.3f} · p homem {p_m:.3f} · razão de probabilidades {p_f / p_m:5.2f}"
          f" · razão de chances {odds(p_f) / odds(p_m):.2f}")
print(f"e^(-coef Sex_male) = {np.exp(-coef['cat__Sex_male']):.2f}")

p mulher 0.601 · p homem 0.093 · razão de probabilidades  6.45 · razão de chances 14.66
p mulher 0.942 · p homem 0.523 · razão de probabilidades  1.80 · razão de chances 14.66
p mulher 0.876 · p homem 0.326 · razão de probabilidades  2.69 · razão de chances 14.66
e^(-coef Sex_male) = 14.66


**Leitura.** A razão de chances mulher/homem dá 14,66 para os três passageiros, exatamente $e^{2{,}685}$. Já a razão de probabilidades vai de 6,45 (homem da 3ª classe, que parte de 0,093) a 1,80 (homem da 1ª classe, que parte de 0,523), com 2,69 para o menino. Por isso o coeficiente se lê como razão de chances: é a medida do efeito que não depende do ponto de partida.

## Passo 8 — o único teste

**Preveja.** A acurácia de teste vai ficar acima ou abaixo dos 80,9% de treino? Muito ou pouco?

In [12]:
from sklearn.metrics import confusion_matrix

print(f"acurácia de teste {model.score(X_test, y_test):.3f} · baseline {dummy.score(X_test, y_test):.3f}")
print(confusion_matrix(y_test, model.predict(X_test), labels=[1, 0]))   # positivo primeiro

acurácia de teste 0.804 · baseline 0.615
[[46 23]
 [12 98]]


**Resposta.** Um pouco abaixo: 0,804 no teste contra 0,809 no treino, meio ponto de diferença, o esperado para um modelo de 9 coeficientes sem nada ajustado. Fica bem acima da baseline (0,615); a matriz mostra 46 dos 69 sobreviventes encontrados e 98 dos 110 mortos reconhecidos.

**Simulação: mexendo no limiar.** O `predict` usa o limiar 0,5. Nas linhas de *treino* (o teste já foi usado), classifique com os limiares 0,3, 0,4, 0,5, 0,6 e 0,7 e imprima, para cada um, a fração dos sobreviventes encontrados, a fração dos mortos reconhecidos e a acurácia.

In [13]:
p_train = model.predict_proba(X_train)[:, 1]          # no treino: o teste já foi usado
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (p_train >= t).astype(int)
    found = pred[y_train == 1].mean()                 # fração dos sobreviventes encontrados
    recog = 1 - pred[y_train == 0].mean()             # fração dos mortos reconhecidos
    print(f"limiar {t}: sobreviventes encontrados {found:.3f} · mortos reconhecidos {recog:.3f}"
          f" · acurácia {(pred == y_train).mean():.3f}")

limiar 0.3: sobreviventes encontrados 0.824 · mortos reconhecidos 0.745 · acurácia 0.775
limiar 0.4: sobreviventes encontrados 0.751 · mortos reconhecidos 0.804 · acurácia 0.784
limiar 0.5: sobreviventes encontrados 0.714 · mortos reconhecidos 0.868 · acurácia 0.809
limiar 0.6: sobreviventes encontrados 0.652 · mortos reconhecidos 0.916 · acurácia 0.815
limiar 0.7: sobreviventes encontrados 0.553 · mortos reconhecidos 0.964 · acurácia 0.806


**Leitura.** Subir o limiar de 0,3 para 0,7 troca sobreviventes encontrados (de 0,824 para 0,553) por mortos reconhecidos (de 0,745 para 0,964). A acurácia mal se mexe, entre 0,775 e 0,815 (máximo em 0,6), e esconde essa troca: o limiar certo depende do custo de cada tipo de erro, não da acurácia.

**Para fechar a aula.** Em uma frase cada: (1) o que um coeficiente da regressão logística multiplica? (2) Por que o menino da simulação do passo 5 recebeu uma probabilidade tão baixa? (3) Por que o conjunto de teste só pode ser usado uma vez?

**Resposta.**

1. Um coeficiente multiplica a *chance* $p/(1-p)$: uma unidade a mais do atributo, com o resto fixo, multiplica a chance por $e^{\text{coef}}$ (equivalentemente, soma o coeficiente ao log da chance).
2. Porque o modelo é aditivo e aplica ao menino o mesmo peso de `Sex_male` dos adultos (chance dividida por 14,66), sem saber que no treino todas as crianças da 2ª classe sobreviveram.
3. Porque cada vez que olhamos o teste e mudamos algo por causa dele a estimativa fica otimista: o teste deixa de representar passageiros que o modelo nunca viu.

## Sua vez

Para casa, com mais tempo:

1. **A interação.** O passo 1 mostrou que a terceira classe prejudica as mulheres muito mais que os homens. Acrescente a coluna `df['female_3rd'] = ((df['Sex'] == 'female') & (df['Pclass'] == 3)).astype(int)`, coloque-a em `features` e em `categorical` (com `drop='first'`, seu coeficiente compara 1 com 0) e refaça os passos 2 a 7. Leia a razão de chances dela em palavras. Como muda o coeficiente `Sex_male`? Por quê? Para decidir se a coluna ajuda, divida `X_train` de novo em uma parte de treino e uma de validação e compare os dois modelos ali, não no teste.
2. **O limiar.** Nessa validação, calcule `p = model.predict_proba(X_val)[:, 1]` e conte os sobreviventes perdidos e os alarmes falsos para os limiares 0,3, 0,5 e 0,7. Que limiar você usaria se perder um sobrevivente fosse três vezes pior que um alarme falso?
3. **Por que uma categoria de referência.** Monte o pipeline de novo com `OneHotEncoder()` (sem `drop`) e ajuste-o. Depois some 5 aos três pesos de `Pclass` em `model[-1].coef_` e subtraia 5 de `model[-1].intercept_`: as probabilidades previstas em `X_test` mudam? O que isso diz sobre ler um único coeficiente quando toda categoria mantém sua coluna?
4. **Do zero.** Tome `Xt = model[0].transform(X_train)` e rode o laço de gradiente descendente desta aula (pesos e intercepto, \(\eta = 0{,}5\)). Compare com `model[-1].coef_` e `model[-1].intercept_`. Depois de quantas iterações eles concordam em duas casas decimais?

### Sua vez — exercício 1: a interação

In [14]:
df['female_3rd'] = ((df['Sex'] == 'female') & (df['Pclass'] == 3)).astype(int)
features_int = features + ['female_3rd']
categorical_int = categorical + ['female_3rd']
Xi_train, Xi_test = df.loc[X_train.index, features_int], df.loc[X_test.index, features_int]  # mesma divisão do passo 2


def build_model(cat_cols):
    pre = ColumnTransformer([
        ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
        ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                              OneHotEncoder(drop='first')), cat_cols)])
    return make_pipeline(pre, LogisticRegression(C=np.inf, max_iter=1000))


model_int = build_model(categorical_int).fit(Xi_train, y_train)
coef_int = pd.Series(model_int[-1].coef_[0], index=model_int[0].get_feature_names_out())
print(f"{coef_int.size} coeficientes · acurácia de treino {model_int.score(Xi_train, y_train):.3f}"
      f" (sem a interação {model.score(X_train, y_train):.3f})")
print(pd.DataFrame({'sem interação': coef.reindex(coef_int.index), 'com interação': coef_int,
                    'e^coef (com)': np.exp(coef_int)}).round(3))

10 coeficientes · acurácia de treino 0.819 (sem a interação 0.809)
                   sem interação  com interação  e^coef (com)
num__Age                  -0.513         -0.616         0.540
num__SibSp                -0.267         -0.284         0.753
num__Parch                -0.072         -0.006         0.994
num__Fare                  0.108          0.022         1.022
cat__Pclass_2             -0.987         -1.439         0.237
cat__Pclass_3             -2.239         -1.740         0.176
cat__Sex_male             -2.685         -4.342         0.013
cat__Embarked_Q            0.320          0.404         1.498
cat__Embarked_S           -0.417         -0.495         0.610
cat__female_3rd_1            NaN         -2.634         0.072


In [15]:
# passos 5 e 6: os dois passageiros e uma mulher da 3ª classe, com e sem a interação
trio = pd.concat([passengers, passengers.iloc[[0]].assign(Sex='female')], ignore_index=True)
trio['female_3rd'] = ((trio['Sex'] == 'female') & (trio['Pclass'] == 3)).astype(int)
print(pd.DataFrame({'Pclass': trio['Pclass'], 'Sex': trio['Sex'],
                    'p sem interação': model.predict_proba(trio[features])[:, 1],
                    'p com interação': model_int.predict_proba(trio)[:, 1]}).round(3))
print(f"mulher vs homem, fora da 3ª classe: chance x {np.exp(-coef_int['cat__Sex_male']):.2f}")
print(f"mulher vs homem, na 3ª classe:      chance x "
      f"{np.exp(-coef_int['cat__Sex_male'] + coef_int['cat__female_3rd_1']):.2f}")

   Pclass     Sex  p sem interação  p com interação
0       3    male            0.093            0.134
1       1  female            0.942            0.986
2       3  female            0.601            0.462
mulher vs homem, fora da 3ª classe: chance x 76.86
mulher vs homem, na 3ª classe:      chance x 5.52


In [16]:
from sklearn.metrics import log_loss

# a decisão: treino/validação dentro de X_train (o teste continua intocado)
Xi_fit, Xi_val, y_fit, y_val = train_test_split(Xi_train, y_train, test_size=0.25,
                                                stratify=y_train, random_state=42)
model_base = build_model(categorical).fit(Xi_fit[features], y_fit)
model_val = build_model(categorical_int).fit(Xi_fit, y_fit)
for name, m, B in [('sem interação', model_base, Xi_val[features]),
                   ('com interação', model_val, Xi_val)]:
    print(f"{name}: acurácia de validação {m.score(B, y_val):.3f}"
          f" · log loss {log_loss(y_val, m.predict_proba(B)[:, 1]):.3f}")

sem interação: acurácia de validação 0.775 · log loss 0.446

com interação: acurácia de validação 0.798 · log loss 0.425


**Conclusão.** `female_3rd` tem coeficiente −2,634, razão de chances 0,072: ser mulher *e* estar na 3ª classe multiplica a chance por 0,07 além do que sexo e classe já preveem separadamente. `Sex_male` passa de −2,685 para −4,342 porque, sem a interação, ele era uma média entre a enorme vantagem feminina na 1ª e na 2ª classe e a vantagem menor na 3ª; agora mede só a primeira (chance ×76,9 fora da 3ª classe, contra ×5,5 dentro dela), e a mulher da 3ª classe cai de 0,601 para 0,462, mais perto dos 50% do passo 1. Na validação (178 linhas separadas de `X_train`), o modelo com interação ganha: acurácia 0,798 contra 0,775 e log loss 0,425 contra 0,446. A coluna ajuda, com a ressalva de que 2,3 pontos são cerca de 4 passageiros.

### Sua vez — exercício 2: o limiar

In [17]:
p = model_val.predict_proba(Xi_val)[:, 1]
for t in [0.3, 0.5, 0.7]:
    pred = (p >= t).astype(int)
    missed = ((pred == 0) & (y_val == 1)).sum()       # sobreviventes perdidos
    false_alarm = ((pred == 1) & (y_val == 0)).sum()  # alarmes falsos
    print(f"limiar {t}: perdidos {missed:3d} · alarmes falsos {false_alarm:3d}"
          f" · custo 3*perdidos + alarmes = {3 * missed + false_alarm}")

limiar 0.3: perdidos  13 · alarmes falsos  23 · custo 3*perdidos + alarmes = 62
limiar 0.5: perdidos  24 · alarmes falsos  12 · custo 3*perdidos + alarmes = 84
limiar 0.7: perdidos  31 · alarmes falsos   4 · custo 3*perdidos + alarmes = 97


**Conclusão.** Do limiar 0,3 para 0,7, os sobreviventes perdidos vão de 13 para 31 e os alarmes falsos de 23 para 4. Se perder um sobrevivente custa três alarmes falsos, os custos são 62, 84 e 97: eu usaria 0,3. Isso bate com a teoria: com custos 3 para 1 e probabilidades bem calibradas, o limiar ótimo é $1/(1+3) = 0{,}25$, abaixo de 0,5.

### Sua vez — exercício 3: por que uma categoria de referência

In [18]:
preprocess_full = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder()), categorical)])
model_full = make_pipeline(preprocess_full, LogisticRegression(C=np.inf, max_iter=1000))
model_full.fit(X_train, y_train)

names_full = model_full[0].get_feature_names_out()
pclass_cols = [i for i, n in enumerate(names_full) if n.startswith('cat__Pclass_')]
p_before = model_full.predict_proba(X_test)[:, 1]
print("pesos de Pclass antes:", model_full[-1].coef_[0, pclass_cols].round(3),
      f"· intercepto {model_full[-1].intercept_[0]:.3f}")
model_full[-1].coef_[0, pclass_cols] += 5
model_full[-1].intercept_ -= 5
p_after = model_full.predict_proba(X_test)[:, 1]
print("pesos de Pclass depois:", model_full[-1].coef_[0, pclass_cols].round(3),
      f"· intercepto {model_full[-1].intercept_[0]:.3f}")
print(f"maior mudança nas probabilidades de X_test: {np.abs(p_after - p_before).max():.2e}")

pesos de Pclass antes: [ 1.128  0.137 -1.113] · intercepto 0.153


pesos de Pclass depois: [6.128 5.137 3.887] · intercepto -4.847
maior mudança nas probabilidades de X_test: 3.33e-16


**Conclusão.** As probabilidades em `X_test` não mudam (maior diferença $3 \times 10^{-16}$, arredondamento): todo passageiro tem exatamente um 1 nas três colunas de `Pclass`, então somar 5 aos três pesos soma 5 a todo escore, e o intercepto −5 desfaz isso. Há infinitos conjuntos de pesos com as mesmas previsões, e um coeficiente sozinho (por exemplo, 1,128 para a 1ª classe) não tem significado; só as diferenças entre categorias têm. Com `drop='first'`, a categoria de referência fixa esse grau de liberdade e cada coeficiente passa a ser lido contra ela.

### Sua vez — exercício 4: do zero

In [19]:
Xt = np.asarray(model[0].transform(X_train))
yt = y_train.to_numpy()
w_sk, b_sk = model[-1].coef_[0], model[-1].intercept_[0]
sigmoid = lambda z: 1 / (1 + np.exp(-z))

w_gd, b_gd, eta = np.zeros(Xt.shape[1]), 0.0, 0.5
agree = None
for it in range(1, 50001):
    p = sigmoid(Xt @ w_gd + b_gd)
    w_gd -= eta * Xt.T @ (p - yt) / len(yt)
    b_gd -= eta * np.mean(p - yt)
    gap = max(np.abs(w_gd - w_sk).max(), abs(b_gd - b_sk))
    if it in (10, 100, 1000, 10000, 50000):
        print(f"iter {it:5d}  maior diferença para o scikit-learn {gap:.4f}")
    if agree is None and gap < 0.005:
        agree = it
print(f"concordam em duas casas decimais (diferença < 0,005) a partir da iteração {agree}")
print(pd.DataFrame({'gradiente': np.append(w_gd, b_gd), 'scikit-learn': np.append(w_sk, b_sk)},
                   index=list(names) + ['intercepto']).round(4))

iter    10  maior diferença para o scikit-learn 2.8186
iter   100  maior diferença para o scikit-learn 1.7678
iter  1000  maior diferença para o scikit-learn 0.1494


iter 10000  maior diferença para o scikit-learn 0.0011

iter 50000  maior diferença para o scikit-learn 0.0011
concordam em duas casas decimais (diferença < 0,005) a partir da iteração 2294
                 gradiente  scikit-learn
num__Age           -0.5133       -0.5132
num__SibSp         -0.2673       -0.2673
num__Parch         -0.0717       -0.0719
num__Fare           0.1079        0.1080
cat__Pclass_2      -0.9867       -0.9869
cat__Pclass_3      -2.2397       -2.2394
cat__Sex_male      -2.6849       -2.6849
cat__Embarked_Q     0.3200        0.3199
cat__Embarked_S    -0.4182       -0.4172
intercepto          2.7830        2.7822


**Conclusão.** Com $\eta = 0{,}5$ e pesos iniciais zero, a maior diferença para o scikit-learn cai de 2,82 (iteração 10) para 0,149 (iteração 1.000) e fica abaixo de 0,005 a partir da iteração 2.294: daí em diante os dois concordam em duas casas decimais. Depois de 10.000 iterações a diferença trava em 0,0011, que vem da tolerância de parada do scikit-learn, não do gradiente (que já convergiu).